# Image Filtering Notebook

This notebook is designed for **self-paced CV learning** and focuses on Image Filtering. It includes:

- Core concept explanations
- Code examples
- Runnable experiments
- Guided tasks
- Reflection and summary

---

## Learning Objectives

After completing this notebook, you should be able to:

1. Understand the mathematical representation of images  
2. Master the basic idea of point processing  
3. Understand convolution and filtering  
4. Implement and compare Box Filter and Gaussian Filter  
5. Understand separable filters  
6. Use Sobel operators for edge detection  
7. Compute image gradients and analyze results  
8. Understand why smoothing before differentiation is necessary


## 0. Environment Setup

Please make sure the following libraries are installed:

- numpy
- matplotlib
- opencv-python

If you run this locally, you can uncomment the next cell to install them.


In [ ]:
# If dependencies are missing in your local environment, uncomment and run
# !pip install numpy matplotlib opencv-python


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import time

plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['image.cmap'] = 'gray'

def show_image(img, title="", cmap=None):
    plt.figure()
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap or 'gray')
    else:
        plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()

def show_two_images(img1, title1, img2, title2, cmap1=None, cmap2=None):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    if img1.ndim == 2:
        plt.imshow(img1, cmap=cmap1 or 'gray')
    else:
        plt.imshow(img1)
    plt.title(title1)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    if img2.ndim == 2:
        plt.imshow(img2, cmap=cmap2 or 'gray')
    else:
        plt.imshow(img2)
    plt.title(title2)
    plt.axis('off')
    plt.show()


## 1. Load an Image

You can use your own image, or start by loading one as shown below.

If the current directory does not contain `sample.jpg`, the notebook will generate a test image automatically.

In [ ]:
img = cv2.imread('sample.jpg')

if img is None:
    # Generate a simple test image
    img = np.zeros((256, 256, 3), dtype=np.uint8)
    cv2.rectangle(img, (40, 40), (120, 200), (255, 255, 255), -1)
    cv2.circle(img, (180, 80), 35, (180, 180, 180), -1)
    cv2.line(img, (140, 180), (230, 220), (255, 255, 255), 4)
    cv2.putText(img, 'CV', (145, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (220, 220, 220), 2)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
show_image(img_rgb, "Original Image")
print("Image shape:", img_rgb.shape)


## Task 1: Understanding Image Representation

Please answer:

1. What do the three dimensions of `img_rgb.shape` represent?
2. Why can a color image be viewed as a 3D tensor?
3. If it becomes a grayscale image, how does the shape change?


## 2. Grayscale Images

In many classic CV tasks, we first convert a color image to grayscale, then perform filtering, edge detection, and related operations.

In [ ]:
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
show_image(gray, "Grayscale Image")
print("Gray image shape:", gray.shape)


## Task 2: Observing Grayscale Images

1. Compared with RGB images, how does the amount of information differ in grayscale images?
2. Why are filtering and edge detection often first demonstrated on grayscale images?


## 3. Point Processing

Point processing means: **each pixel is transformed independently, without using neighboring pixels**.

Typical examples include:

- Brightening / darkening
- Contrast enhancement
- Inversion (invert)


In [ ]:
bright = np.clip(gray.astype(np.int16) + 40, 0, 255).astype(np.uint8)
dark = np.clip(gray.astype(np.int16) - 40, 0, 255).astype(np.uint8)
invert = 255 - gray

show_two_images(bright, "Brightened", dark, "Darkened")
show_image(invert, "Inverted")


In [ ]:
def adjust_contrast_brightness(image, a=1.0, b=0):
    out = a * image.astype(np.float32) + b
    out = np.clip(out, 0, 255)
    return out.astype(np.uint8)

low_contrast = adjust_contrast_brightness(gray, a=0.6, b=0)
high_contrast = adjust_contrast_brightness(gray, a=1.5, b=0)

show_two_images(low_contrast, "Low Contrast", high_contrast, "High Contrast")


## Task 3: Implement Point Processing Yourself

Complete the function below and try different parameters:

- `a > 1`: what happens?
- `0 < a < 1`: what happens?
- `b > 0`: what happens?
- `b < 0`: what happens?


In [ ]:
# TODO: Modify a and b to observe image changes
a = 1.2
b = 20

custom = adjust_contrast_brightness(gray, a=a, b=b)
show_image(custom, f"Custom Transform: a={a}, b={b}")


## 4. Neighborhood Operation

Unlike point processing, neighborhood operations use local neighborhood information around each pixel.

Image filtering is the most typical neighborhood operation:
> Use a kernel (convolution kernel) to compute a weighted sum over each local neighborhood.


## 5. Box Filter (Mean Filter)

A Box Filter replaces each pixel with the average of neighboring pixels, so it produces a **smoothing / blurring** effect.

A common 3x3 Box Filter is:

\[
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
\]


In [ ]:
box_kernel = np.ones((3, 3), dtype=np.float32) / 9.0
box_blur = cv2.filter2D(gray, -1, box_kernel)

show_two_images(gray, "Original Gray", box_blur, "3x3 Box Filter")
print("Box kernel:\n", box_kernel)


## Task 4: Observe the Box Filter

1. What effect does the Box Filter have on the image?
2. Which regions change more noticeably?
3. Why does it create a blur effect?


## 6. Manual Convolution

To truly understand filtering / convolution, we will implement 2D convolution ourselves.

In [ ]:
def manual_convolution(image, kernel):
    kh, kw = kernel.shape
    assert kh % 2 == 1 and kw % 2 == 1, "Kernel size should be odd."

    pad_h = kh // 2
    pad_w = kw // 2

    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    output = np.zeros_like(image, dtype=np.float32)

    # Convolution requires flipping the kernel
    kernel_flipped = np.flipud(np.fliplr(kernel))

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * kernel_flipped)

    output = np.clip(output, 0, 255)
    return output.astype(np.uint8)


In [ ]:
manual_box = manual_convolution(gray, box_kernel)
opencv_box = cv2.filter2D(gray, -1, box_kernel)

show_two_images(manual_box, "Manual Convolution", opencv_box, "OpenCV filter2D")
print("Difference sum:", np.abs(manual_box.astype(np.int32) - opencv_box.astype(np.int32)).sum())


## Task 5: Understanding Convolution

Think about:

1. Why do we flip the kernel during convolution?
2. If we do not flip it, is the operation closer to convolution or correlation?
3. Why do many symmetric kernels (such as Gaussian kernels) appear to produce the same result even without flipping?


## 7. Gaussian Filter

A Gaussian Filter is also a smoothing filter, but instead of simple averaging:

- Pixels closer to the center have larger weights
- Pixels farther from the center have smaller weights

So it usually looks more natural than a Box Filter.

In [ ]:
gaussian_blur = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.0)
show_two_images(box_blur, "Box Filter", gaussian_blur, "Gaussian Filter")


## Task 6: Compare Box Filter and Gaussian Filter

Please observe and answer:

1. Can both filters blur an image?
2. Which type of blur looks more natural to you?
3. Why is Gaussian Filter usually preferred over Box Filter?


## 8. Build a Gaussian Kernel Yourself

Now we manually generate a Gaussian kernel to better understand where the kernel comes from.

In [ ]:
def gaussian_kernel(size=5, sigma=1.0):
    assert size % 2 == 1, "Kernel size should be odd."
    ax = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / np.sum(kernel)
    return kernel.astype(np.float32)

gk = gaussian_kernel(size=5, sigma=1.0)
print("Gaussian kernel:\n", gk)
print("Kernel sum:", gk.sum())

manual_gaussian = manual_convolution(gray, gk)
show_image(manual_gaussian, "Manual Gaussian Blur")


## Task 7: Parameter Experiments

Modify the following parameters and observe the results:

- kernel size: 3, 5, 7, 9
- sigma: 0.5, 1, 2, 3

Think about:

1. What happens to the image as sigma increases?
2. What problems arise if kernel size is too small?
3. Why is a Gaussian kernel usually normalized (sum = 1)?


In [ ]:
size = 7
sigma = 2.0

gk = gaussian_kernel(size=size, sigma=sigma)
blur_custom = manual_convolution(gray, gk)
show_image(blur_custom, f"Gaussian Blur (size={size}, sigma={sigma})")


## 9. Separable Filters

If a 2D filter can be decomposed into:

- one column vector
- one row vector

then it is separable.

This means:
> one 2D convolution = two 1D convolutions

Benefit: **faster computation**.


In [ ]:
g1d = cv2.getGaussianKernel(7, 1.5)  # column vector
sep_blur = cv2.sepFilter2D(gray, -1, g1d, g1d)

show_two_images(cv2.GaussianBlur(gray, (7, 7), 1.5), "GaussianBlur", sep_blur, "Separable Gaussian")


In [ ]:
# Simple speed benchmark
repeats = 50

start = time.time()
for _ in range(repeats):
    _ = cv2.GaussianBlur(gray, (9, 9), 2.0)
t1 = time.time() - start

start = time.time()
g1d = cv2.getGaussianKernel(9, 2.0)
for _ in range(repeats):
    _ = cv2.sepFilter2D(gray, -1, g1d, g1d)
t2 = time.time() - start

print(f"GaussianBlur time: {t1:.4f}s")
print(f"sepFilter2D time: {t2:.4f}s")


## Task 8: Why Are Separable Filters Important?

Please answer:

1. Why are separable filters computationally more efficient?
2. Is Box Filter separable?
3. Is Gaussian Filter separable?
4. If both image size and kernel size are large, why is separability especially valuable?


## 10. Sharpening

The goal of sharpening is to enhance edges and details.

A common idea is:
- keep the original image
- subtract a smoothed result
- or use a specific sharpening kernel


In [ ]:
sharpen_kernel = np.array([[0, -1,  0],
                           [-1, 5, -1],
                           [0, -1,  0]], dtype=np.float32)

sharpened = cv2.filter2D(gray, -1, sharpen_kernel)
show_two_images(gray, "Original", sharpened, "Sharpened")
print("Sharpen kernel:\n", sharpen_kernel)


## Task 9: Observe Sharpening Results

1. Which regions are mainly enhanced by sharpening?
2. Why does sharpening also make noise more obvious?
3. What problems can over-sharpening cause?


## 11. Edge Detection

Edges usually correspond to **strong intensity changes** in an image.

Mathematically, this change can be measured by **derivatives**.

### Finite Difference

For discrete images, we cannot directly compute continuous derivatives, so we use finite-difference approximations.

A common 1D derivative kernel is:

\[
[-1, 0, 1]
\]

In 2D images, we extend this into 2D derivative filters.

## 12. Sobel Filter

Sobel is a classic first-order derivative filter that combines:

- differentiation
- a certain amount of smoothing

Two common directions:

- Sobel X: detects vertical edges
- Sobel Y: detects horizontal edges


In [ ]:
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

show_two_images(np.abs(sobel_x), "Sobel X", np.abs(sobel_y), "Sobel Y")


## Task 10: Understanding Sobel X / Sobel Y

Please answer:

1. Which edge direction does Sobel X respond to more strongly?
2. Which edge direction does Sobel Y respond to more strongly?
3. From the perspective of derivatives, why do edges produce larger responses?


## 13. Image Gradient

We can combine derivatives from two directions into a gradient:

- \( G_x \): change in the x direction
- \( G_y \): change in the y direction

Gradient magnitude:

\[
\sqrt{G_x^2 + G_y^2}
\]

Gradient direction:

\[
\theta = \arctan2(G_y, G_x)
\]


In [ ]:
grad_mag = np.sqrt(sobel_x**2 + sobel_y**2)
grad_dir = np.arctan2(sobel_y, sobel_x)

show_image(grad_mag, "Gradient Magnitude")
print("Gradient direction range:", grad_dir.min(), grad_dir.max())


## Task 11: Gradient Analysis

Please observe the gradient magnitude image and answer:

1. Where is the gradient largest?
2. Why do these locations usually correspond to edges?
3. What is the relationship between gradient direction and edge direction?


## 14. Derivatives Are Sensitive to Noise

Differentiation amplifies high-frequency changes, so it is sensitive to noise.

An important conclusion is:

> **Before edge detection, we usually smooth (blur) first, then take derivatives.**


In [ ]:
# Create a noisy image
noise = np.random.normal(0, 20, gray.shape).astype(np.float32)
noisy = np.clip(gray.astype(np.float32) + noise, 0, 255).astype(np.uint8)

sobel_noisy = cv2.Sobel(noisy, cv2.CV_64F, 1, 0, ksize=3)
blurred_noisy = cv2.GaussianBlur(noisy, (5, 5), 1.0)
sobel_blurred = cv2.Sobel(blurred_noisy, cv2.CV_64F, 1, 0, ksize=3)

show_two_images(noisy, "Noisy Image", np.abs(sobel_noisy), "Sobel on Noisy Image")
show_two_images(blurred_noisy, "Blurred Noisy Image", np.abs(sobel_blurred), "Gaussian + Sobel")


## Task 12: Why Smooth Before Differentiation?

Please answer based on the experiment:

1. Why do many spurious responses appear when Sobel is applied directly to a noisy image?
2. What role does Gaussian Blur play?
3. Does this mean stronger blur is always better? Why?


## 15. Bonus: Laplacian (Second-Order Derivative)

Laplacian can be viewed as the 2D version of a second-order derivative and is also commonly used for edge analysis.

In [ ]:
lap = cv2.Laplacian(gray, cv2.CV_64F)
show_image(np.abs(lap), "Laplacian")


## Task 13 (Bonus)

1. How are Laplacian and Sobel results different?
2. What are the characteristics of first-order and second-order derivatives in edge detection?


## 16. Integrated Practice: Build a Simple Edge Detector

Now connect everything you learned:

1. Read an image
2. Convert to grayscale
3. Gaussian Blur
4. Sobel X / Sobel Y
5. Compute gradient magnitude
6. Apply thresholding to get an edge map


In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 1.0)
gx = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
mag = np.sqrt(gx**2 + gy**2)

threshold = 80
edges = (mag > threshold).astype(np.uint8) * 255

show_two_images(mag, "Gradient Magnitude", edges, f"Edges (threshold={threshold})")


## Final Task: Your Mini Project

Complete a small project and answer the following:

### Part A
Try different parameters:
- Gaussian kernel size
- sigma
- Sobel kernel size
- threshold

### Part B
Choose one of your own images and run the full pipeline.

### Part C
Answer:
1. Which parameters influence edge detection results the most?
2. Why does a too-small threshold produce many noisy edges?
3. Why does a too-large threshold miss real edges?
4. In your image, which edges are easiest to detect, and why?


## Reflection

Summarize in 3-5 sentences:

1. What is point processing?
2. What is filtering?
3. What is the difference between Gaussian Filter and Box Filter?
4. Why is Sobel essentially computing derivatives?
5. Why do we usually smooth before edge detection?


## Extensions (Optional)

If you finish early, you can also try:

- Compare Sobel / Prewitt / Scharr
- Implement separable convolution manually
- Implement unsharp masking
- Try filtering strategies for color images
- Study Laplacian of Gaussian (LoG)
